# Name That LLM: language-model features (GPU extraction)

**Goal:** turn every training passage into numbers a language model "sees": the
**mean-pooled hidden states** of an open LLM, at **every layer**, for three sizes of
**Qwen 3 base** (0.6B, 1.7B, 4B).

Part of the learning-in-public MLOps project
[github.com/Ryqn0/name-that-llm](https://github.com/Ryqn0/name-that-llm), plan P3b.

**Why:** our TF-IDF models are saturated around 0.32 balanced accuracy. A language model's
internal representation captures style in a different way (word choice in context, rhythm),
and early layers are reported to carry the authorship signal best. We'll check that claim
ourselves rather than trust it.

**Why only extraction here:** a GPU is needed to run the LLM, but NOT to choose the best
layer. So this notebook only extracts and saves the features; the layer choice and the
tuning run in CPU notebooks that read this notebook's output. That spends GPU quota only
where a GPU is needed.

## 1. Setup

The package is installed from a git tag, so this notebook always runs the exact code it was
written for. torch and transformers come with Kaggle's GPU image; they are deliberately not
dependencies of our package (ADR 0002).

In [ ]:
%pip install -q "name-that-llm @ git+https://github.com/Ryqn0/name-that-llm@v0.5.0"

In [ ]:
import gc
import time

import numpy as np
import torch
import transformers
from packaging.version import Version

from name_that_llm.data import load_train
from name_that_llm.lm import find_model_dir, load_lm, pooled_hidden_states

print("torch       ", torch.__version__, "| CUDA", torch.version.cuda)
print("transformers", transformers.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

# Qwen 3 support arrived in transformers 4.51. Failing here, in the first seconds, is far
# better than failing after an hour, or in the offline submission.
assert Version(transformers.__version__) >= Version("4.51"), "transformers too old for Qwen 3"
assert torch.cuda.is_available(), "this notebook needs a GPU (accelerator: T4)"

## 2. Training passages, and their order

The features are saved as plain arrays, and other notebooks will match them to labels by
**row position**. Saving the passage ids next to them lets every reader check that the rows
are in the same order. A silent shuffle would make every downstream number meaningless.

In [ ]:
train = load_train()
np.save("/kaggle/working/train_ids.npy", train["id"].to_numpy())
print("train:", train.shape)

## 3. Extract every layer, for each model size

For each passage and each layer, `pooled_hidden_states` averages the per-token hidden
states over the passage's real tokens. Padding never enters the mean, and it's right-padded,
so a passage's vector doesn't depend on its batch. The result for one model is an array of
shape **(layers + 1, passages, hidden size)**, where index 0 is the embedding layer.

The model runs in **float16** on the GPU: half the memory, and the T4's tensor cores make it
fast. float16 has a small range, so we check that every value is finite. If a small model
overflows, it's re-run in float32; the 4B model wouldn't fit in float32, so it would be
skipped instead.

In [ ]:
VARIATIONS = ["0.6b-base", "1.7b-base", "4b-base"]
BATCH_SIZE, MAX_LENGTH = 16, 512
texts = train["text"].tolist()

for variation in VARIATIONS:
    start = time.time()
    model_dir = find_model_dir(variation)  # searched, never hard-coded: Kaggle paths move
    model, tokenizer = load_lm(model_dir)
    feats = pooled_hidden_states(
        texts, model, tokenizer, batch_size=BATCH_SIZE, max_length=MAX_LENGTH
    )

    if not np.isfinite(feats).all():
        del model
        gc.collect()
        torch.cuda.empty_cache()
        if variation == "4b-base":
            print(f"{variation}: non-finite values in float16 and too big for float32: skipped")
            continue
        print(f"{variation}: non-finite values in float16, re-running in float32")
        model, tokenizer = load_lm(model_dir, dtype="float32")
        feats = pooled_hidden_states(
            texts, model, tokenizer, batch_size=BATCH_SIZE, max_length=MAX_LENGTH
        )
        assert np.isfinite(feats).all(), f"{variation}: still non-finite in float32"

    # Stored as float16 to halve the file size. Check again: a value beyond float16's range
    # (about 65,000) would turn into infinity when cast.
    small = feats.astype(np.float16)
    to_save = small if np.isfinite(small).all() else feats
    np.save(f"/kaggle/working/features-{variation}.npy", to_save)
    print(f"{variation}: {to_save.shape} {to_save.dtype}, {(time.time() - start) / 60:.1f} min")

    del model, feats, small, to_save
    gc.collect()
    torch.cuda.empty_cache()  # free the GPU before loading the next, bigger model

## Outputs

- `features-<variation>.npy`: (layers + 1, 2699, hidden), mean-pooled hidden states
- `train_ids.npy`: the row order of those arrays

The next notebook, **05-lm-sweep** (CPU), reads them, cross-validates a logistic regression
on every layer with our prompt-grouped folds, and blends the best one with the TF-IDF model.